# 📸 Gruppenfoto mit dem JetBot

In diesem Notebook steuert ihr den JetBot mit dem Gamepad und macht mit der **A-Taste** ein Foto.

| Eingabe | Aktion |
|---------|--------|
| linker Stick hoch/runter | linkes Rad |
| rechter Stick hoch/runter | rechtes Rad |
| **A** | Foto aufnehmen und speichern |

> ⚠️ Vorher in allen anderen Notebooks `camera.stop()` ausführen und die Kernel beenden.

## 1. Gamepad verbinden

Gamepad am **Laptop** einstecken (Xbox-360-Modus), dann die Zelle ausführen und **eine Taste drücken**, bis das Widget Tasten und Achsen anzeigt.

In [ ]:
import ipywidgets.widgets as widgets

controller = widgets.Controller(index=0)  # bei mehreren Gamepads: index=1, 2, ...
display(controller)

## 2. Roboter und Kamera starten

In [ ]:
from jetbot import Robot, Camera, bgr8_to_jpeg
import traitlets

robot = Robot()
camera = Camera.instance(width=640, height=480)

bild = widgets.Image(format='jpeg', width=640, height=480)
kamera_link = traitlets.dlink((camera, 'value'), (bild, 'value'), transform=bgr8_to_jpeg)

display(bild)

## 3. Mit dem Gamepad fahren

Die Sticks werden direkt mit den Motoren verbunden. Das Minus dreht die Richtung um, damit "Stick nach vorne" auch "vorwärts" bedeutet.

In [ ]:
links_link = traitlets.dlink((controller.axes[1], 'value'), (robot.left_motor, 'value'), transform=lambda x: -x)
rechts_link = traitlets.dlink((controller.axes[3], 'value'), (robot.right_motor, 'value'), transform=lambda x: -x)

## 4. Foto mit der A-Taste

Tragt euren Gruppenbuchstaben ein. Jedes Foto wird mit Uhrzeit im Ordner `fotos/` gespeichert, so geht kein Foto verloren.

In [ ]:
import os
from datetime import datetime

GRUPPE = "a"          # <-- euren Gruppenbuchstaben eintragen
ORDNER = "fotos"
os.makedirs(ORDNER, exist_ok=True)

vorschau = widgets.Image(format='jpeg', width=320, height=240)
status = widgets.Label("Drückt A für ein Foto 📸")

def foto_machen(change):
    if change['new']:  # nur beim Drücken, nicht beim Loslassen
        zeit = datetime.now().strftime("%H-%M-%S")
        dateiname = os.path.join(ORDNER, f"gruppe_{GRUPPE}_{zeit}.jpg")
        jpeg = bgr8_to_jpeg(camera.value)
        with open(dateiname, 'wb') as f:
            f.write(jpeg)
        vorschau.value = jpeg
        status.value = f"Gespeichert: {dateiname}"

controller.buttons[0].observe(foto_machen, names='pressed')  # buttons[0] = A

display(widgets.VBox([status, vorschau]))

## 5. Foto ins Repository bringen

1. Im Jupyter-Dateibrowser links den Ordner `fotos/` öffnen
2. Bestes Foto auswählen → Rechtsklick → **Download**
3. Umbenennen in `gruppe_<buchstabe>.jpg`, z. B. `gruppe_a.jpg`
4. In den Ordner `gruppenfotos/` im Repository legen und per Pull Request hochladen

## 6. Aufräumen

Vor dem Wechsel in ein anderes Notebook unbedingt ausführen!

In [ ]:
controller.buttons[0].unobserve(foto_machen, names='pressed')
links_link.unlink()
rechts_link.unlink()
kamera_link.unlink()
robot.stop()
camera.stop()
print("Alles gestoppt ✅")